# EDO Sense Soroll

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from typing import Tuple

Algorisme de l'article de Strahan et al. 2023,
a partir de trajectòries del sistema dinàmic donat 
pel següent sistema d'EDOs, on $f(t)=sin(t)$.
$$x'=y + \epsilon f(t)$$ $$y'=x-x^3$$


In [3]:
def edo(t, z, epsilon=0.05, funcio_soroll=np.sin):
    """Paràmetres:
        t: temps
        z: posició al pla R^2
        epsilon: paràmetre entre 0 i 1 que regula el soroll
        funcio_soroll: pertorbació que depèn del temps
    Retorna el camp vectorial del sistema d'EDOs autònom x'=y; y'=x-x^3.
    """
    x, y = z
    soroll = epsilon * funcio_soroll(t)
    camp_vectorial = [y + soroll, x - x**3]
    return np.array(camp_vectorial)

Paràmetres

In [6]:
t_span = [0, 20]
t_steps = 2000
t_valors = np.linspace(t_span[0], t_span[1], t_steps)
condicions_inicials = [[0.3, 0.3]]

# Algorisme de l'article de Strahan et al. 2023

In [24]:
radi = 0.3
centre_esquerre = (-1, 0)
centre_dret = (1, 0)
resolucio_estandard = 2000

In [23]:
def pertany_a_circumferencia(x: float,
                             y: float, 
                             centre: Tuple[float, float], 
                             radi: float
                             ) -> bool:
    a, b = centre
    return (x-a)**2 + (y-b)**2 <= radi**2

In [11]:
def pertany_a_la_regio_A(x: float, y: float) -> bool: # Punt fix esquerre = (-1,0)
    return pertany_a_circumferencia(x, y, centre_esquerre, radi) 

def pertany_a_la_regio_B(x: float, y: float) -> bool: # Punt fix dret = (1,0)
    return pertany_a_circumferencia(x, y, centre_dret, radi)

In [ ]:

def grafica_circumferencia(centre: Tuple[float, float], radi: float, 
                           nom_regio:str, resolucio: int = resolucio_estandard):
    theta = np.linspace(0, 2*np.pi, resolucio)
    a, b = centre
    xx = radi*np.cos(theta) + a
    yy = radi*np.sin(theta)
    plt.plot(xx, yy + b, 'black')
    plt.plot(xx, - yy + b, 'black')
    plt.annotate(nom_regio, centre)

def grafica_regio_A():
    grafica_circumferencia(centre_esquerre, radi, nom_regio='A')

def grafica_regio_B():
    grafica_circumferencia(centre_dret, radi, nom_regio='B')

In [18]:
def grafica_solucio(edo, t_span, t_valors, condicions_inicials, desa_pdf=False):
    grafica_regio_A()
    grafica_regio_B()
    for ci in condicions_inicials:
        sol = solve_ivp(edo, t_span, ci, t_eval=t_valors)
        plt.plot(sol.y[0], sol.y[1], color = 'orange')
        inici_string = f'Inici = ({ci[0]:.2f},{ci[1]:.2f})'
        plt.plot(ci[0], ci[1], '-o', label = inici_string)
        final = (sol.y[0][-1], sol.y[1][-1]) #Posició final
        final_string = f'Final = ({final[0]:.2f},{final[1]:.2f})'
        plt.plot(final[0], final[1], '-o', color = 'red', label = final_string)
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(); plt.grid()
    plt.gca().set_aspect('equal')
    if desa_pdf:
        plt.savefig('edo.pdf')
    plt.show()

In [10]:
x_min, x_max = (-2, 2)
y_min, y_max = (-1.5, 1.5)